# Big Data Analytics: Sentiment Analysis of Amazon Grocery & Gourmet Food Reviews

**Course:** CAP 4786 - Topics in Big Data  
**Project:** Sentiment Analysis of Customer Reviews  
**Dataset:** Amazon Reviews 2023 (Grocery and Gourmet Food Category - 14.3M reviews)

This notebook contains the complete PySpark pipeline for loading, preprocessing, modeling, and evaluating sentiment on a large-scale dataset of Amazon customer reviews. It is designed to be run on a Google Cloud Platform (GCP) Dataproc cluster.

## 1. Setup and Initialization

Install a specific version of the Hugging Face `datasets` library (2.14.7) that supports streaming with `trust_remote_code`, allowing us to stream only the rows we need without downloading the full dataset file.

In [ ]:
!pip install "datasets==2.14.7" -q

Initialize the PySpark session and import necessary libraries.

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, lower, regexp_replace
from pyspark.sql.types import StringType, FloatType
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
from datasets import load_dataset
import pandas as pd
import time

# SparkSession is auto-created in Dataproc Jupyter, but we include this for safety
spark = SparkSession.builder \
    .appName('Amazon_Grocery_Sentiment_Analysis') \
    .getOrCreate()

print(f'Spark Version: {spark.version}')

## 2. Data Acquisition (Streaming from Hugging Face)

We stream a subset of the Grocery and Gourmet Food category reviews directly via the Hugging Face API. No file download or cloud storage is required — only the rows we need are loaded into memory.

To run the scalability test, change `subset_size` to a larger number (e.g., 2000000 or 5000000) and re-run this cell and all cells below.

In [ ]:
print('Streaming Grocery and Gourmet Food reviews from Hugging Face...')
start_time = time.time()

# Stream the dataset — only the rows we request are transferred, not the full 14.3M row file
dataset = load_dataset(
    'McAuley-Lab/Amazon-Reviews-2023',
    'raw_review_Grocery_and_Gourmet_Food',
    split='full',
    streaming=True,
    trust_remote_code=True
)

# Adjust this number for your scalability tests: 500000 / 2000000 / 5000000
subset_size = 500000

# Stream data into a list, keeping only rating and review text
data_list = []
for i, row in enumerate(dataset):
    if i >= subset_size:
        break
    if row['text'] is not None and row['rating'] is not None:
        data_list.append({'rating': float(row['rating']), 'text': str(row['text'])})

# Convert to Spark DataFrame via Pandas
pdf = pd.DataFrame(data_list)
df = spark.createDataFrame(pdf)

end_time = time.time()
print(f'Loaded {df.count()} rows in {end_time - start_time:.2f} seconds.')
df.show(5, truncate=True)

## 3. Data Preprocessing & Feature Engineering

We clean the review text and create a binary sentiment label from the star rating.

**Baseline Mapping (star rating as sentiment proxy):**
- Ratings 4.0 - 5.0 -> Positive (Label 1)
- Ratings 1.0 - 2.0 -> Negative (Label 0)
- Rating 3.0 -> Neutral (filtered out for a cleaner binary classification task)

In [ ]:
# 1. Filter out neutral reviews (rating == 3.0)
df_filtered = df.filter(col('rating') != 3.0)

# 2. Create binary sentiment label from star rating (this is our baseline)
df_labeled = df_filtered.withColumn('label', when(col('rating') >= 4.0, 1).otherwise(0))

# 3. Clean text: lowercase and remove punctuation/special characters
df_clean = df_labeled.withColumn('clean_text', lower(col('text')))
df_clean = df_clean.withColumn('clean_text', regexp_replace(col('clean_text'), '[^a-zA-Z\\s]', ''))

# Check class distribution to identify any data skew
print('Class Distribution (0 = Negative, 1 = Positive):')
df_clean.groupBy('label').count().orderBy('label').show()

## 4. Spark ML Pipeline (TF-IDF + Logistic Regression)

We build a Spark ML pipeline with the following stages:
1. **Tokenizer** - splits review text into individual words
2. **StopWordsRemover** - removes common words like 'the', 'is', 'and'
3. **HashingTF** - computes Term Frequency (TF) for each word
4. **IDF** - computes Inverse Document Frequency to weight rare, meaningful words higher
5. **Logistic Regression** - trains the sentiment classifier on the TF-IDF features

In [ ]:
# Define each stage of the pipeline
tokenizer = Tokenizer(inputCol='clean_text', outputCol='words')
remover = StopWordsRemover(inputCol='words', outputCol='filtered_words')
hashingTF = HashingTF(inputCol='filtered_words', outputCol='rawFeatures', numFeatures=10000)
idf = IDF(inputCol='rawFeatures', outputCol='features')
lr = LogisticRegression(maxIter=10, regParam=0.01, featuresCol='features', labelCol='label')

# Assemble the full pipeline
pipeline = Pipeline(stages=[tokenizer, remover, hashingTF, idf, lr])

# Split into 80% training and 20% testing
train_data, test_data = df_clean.randomSplit([0.8, 0.2], seed=42)
print(f'Training Data Count: {train_data.count()}')
print(f'Testing Data Count:  {test_data.count()}')

## 5. Model Training

Spark distributes the training workload across the cluster nodes. Record the training time printed below — you will use this for your scalability comparison chart.

In [ ]:
print('Training the model — this may take a few minutes...')
train_start = time.time()

model = pipeline.fit(train_data)

train_end = time.time()
print(f'Model training completed in {train_end - train_start:.2f} seconds.')

## 6. Model Evaluation

We evaluate the trained model on the held-out test set using Accuracy and F1-Score. These are the metrics you will report in your final project report.

In [ ]:
# Generate predictions on the test set
predictions = model.transform(test_data)

# Show a sample of predictions
predictions.select('clean_text', 'label', 'prediction', 'probability').show(5, truncate=60)

# Calculate Accuracy
evaluator_acc = MulticlassClassificationEvaluator(
    labelCol='label', predictionCol='prediction', metricName='accuracy'
)
accuracy = evaluator_acc.evaluate(predictions)

# Calculate F1-Score
evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol='label', predictionCol='prediction', metricName='f1'
)
f1_score = evaluator_f1.evaluate(predictions)

print(f'Test Accuracy : {accuracy * 100:.2f}%')
print(f'Test F1-Score : {f1_score * 100:.2f}%')

## 7. Scalability Test

To demonstrate the 'Big Data' aspect of this project, run this notebook three times with different values of `subset_size` in Section 2 and record the training time each time:

| Run | subset_size | Training Time (seconds) |
|-----|-------------|-------------------------|
| 1   | 500,000     | *(record here)*         |
| 2   | 2,000,000   | *(record here)*         |
| 3   | 5,000,000   | *(record here)*         |

Plot these three data points as a line chart (x = dataset size, y = training time) for your final presentation. This directly demonstrates how Spark scales with data volume.